In [1]:
import geemap
import ee

Map = geemap.Map()
Map

TransportError: HTTPSConnectionPool(host='oauth2.googleapis.com', port=443): Max retries exceeded with url: /token (Caused by ProxyError('Unable to connect to proxy', RemoteDisconnected('Remote end closed connection without response')))

In [ ]:
image = ee.Image("projects/ee-2431566134liumonarch/assets/Gansu_pv_2020_part3_map")
Map.addLayer(image, {'palette':"blue"},'pv_final')

In [ ]:
roi = image.geometry()
Map.addLayer(roi, {}, "roi")

In [ ]:
geemap.download_ee_image(image, r'D:\Work\Gansu_pv_2020_part3_map.tif', region=roi, crs='epsg:4326', scale=30)

In [ ]:
Map.center_object(roi, 8)

### 连通像元数量去除小斑块

In [ ]:
img_c = image.connectedPixelCount(50, False)
image_mask = image.mask(img_c.gte(50))
Map.addLayer(image_mask, {'palette':"red"},'pv_final_mask')

In [ ]:
image_mask_vector = image_mask.reduceToVectors(**{
    'geometry':roi,
    'scale': 30,
    'maxPixels': 1e13
})

In [ ]:
Map.addLayer(image_mask_vector, {'color':'green'}, 'vector')

### 面积周长比去除不规则矢量

In [ ]:
def feature_area_per(feature):
    feature = ee.Feature(feature)
    area = feature.area(0.01)
    perimeter = feature.perimeter(0.01)
    feature = feature.set('area', area, 'perimeter', perimeter, 'ratio', area.divide(perimeter))
    return feature

In [ ]:
feature_image = image_mask_vector.map(feature_area_per)

In [ ]:
Map.addLayer(feature_image.filter(ee.Filter.gt('ratio', 45)), 
             {'color':'red'}, 'feature_image')

### 矢量导出

In [ ]:
geemap.ee_to_shp(image_mask_vector, r'D:\Data\feature_image.shp')